# MNIST Digit Classifier — CNN Model Training

## 1. Import dependencies

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import numpy as np
import matplotlib.pyplot as plt

## 2. Load the MNIST dataset
Will download automatically if not present.


In [ ]:
# train data will be pixel values 0-255, and train labels will be the corresponding digit of the image (0-9)
(train_data, train_labels), (test_data, test_labels) = datasets.mnist.load_data()

## 3. Get a feel for the data
Print out various information on the data (shape, first entry).

In [ ]:
# Display dataset shapes
print(f"Training data shape: {train_data.shape}, Labels shape: {train_labels.shape}")
print(f"Testing data shape: {test_data.shape}, Labels shape: {test_labels.shape}")

# Print the first entry of the data.
print (f"Label: {train_labels[0]}")
print (f"Pixel data: {train_data[0]}")

## 4. Preprocess the data
Normalize the pixels and add a new dimension to represent the number of pixel channels (1 for grayscale).

In [ ]:
max_pixel_value = 255.0

# Normalize the pixel values to the range [0, 1]. This is necessary as models will multiply values by wieghts, 
# and high pixel values can lead to large gradients and unstable training. It also helps improve performance of the model.
#print (f"test_data[0] before normalization: {test_data[0]}")
train_data = train_data / max_pixel_value
test_data = test_data / max_pixel_value
#print (f"test_data[0] after normalization: {test_data[0]}")

# Because I will be using TensorFlow's Conv2D layer, the input data needs to be in the shape (num_samples, height, width, channels). 
# The MNIST dataset is currently in the shape (num_samples, height, width), so I need to add an extra dimension for the channels (which is 1 for grayscale images).
train_data = train_data[..., np.newaxis]
test_data = test_data[..., np.newaxis]

## 5. Build the Convolutional Neural Network (CNN)
This model uses three convolutional layers followed by a dense classifier.

In [ ]:
image_width = 28
image_height = 28
num_channels = 1

model = models.Sequential()

# The input shape is (28, 28, 1) for the Conv2D layer, which corresponds to the height, width, and number of channels of the input images.
model.add(layers.Input(shape=(image_width, image_height, num_channels)))

# Use 32 filters as apparently that's a common choice for the first layer of a CNN for it to learn low level features. 
# The kernel size of (3, 3) is also a common choice as it allows the model to learn local patterns in the images.
# The activation function 'relu' is used to introduce non-linearity into the model, which helps it learn more complex patterns like images.
model.add(layers.Conv2D(32, (3, 3), activation='relu'))

# Downsample the feature maps by a factor of 2 using max pooling. 
# This helps reduce the spatial dimensions of the feature maps, which can help reduce overfitting and improve computational efficiency.
model.add(layers.MaxPooling2D((2, 2)))

# 64 filters are used in the second Conv2D layer to allow the model to learn more complex features from the images.
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# Flatten the 3D feature maps to 1D feature vectors, which can be fed into fully connected layers.
model.add(layers.Flatten())

# Feeds the output from the convolutional base into Dense layers to perform classification
# The first Dense layer with 64 units and 'relu' activation allows the model to learn complex relationships between the features extracted by the convolutional layers.
model.add(layers.Dense(64, activation='relu'))
# The final Dense layer with 10 units and 'softmax' activation is used to output the probabilities for each of the 10 classes (digits 0-9).
model.add(layers.Dense(10, activation='softmax'))

# Display the model architecture
model.summary()

## 6. Compile the model.

In [ ]:
# The 'adam' optimizer updates the model’s weights during training based on the computed gradients, which helps the model learn from the data.
# The 'sparse_categorical_crossentropy' loss function is used for multi-class classification problems where the labels are integers. 
# It measures the difference between the predicted probabilities and the true labels, and the model will try to minimize this loss during training.
# The 'accuracy' metric is used to evaluate the performance of the model during training and testing. It calculates the percentage of correctly classified samples.
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## 7. Train the model
Train the model on the training data until we detect that it is beginning to overfit, afterwhich we will use the epoch before overfitting occurred.

In [ ]:
# This will create a function that will stop the model from overfitting during training by detectiing when the validation loss stops 
# improving for 2 consecutive epochs and then restores the model weights from the epoch with the best validation loss.
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Fits the model to the training data.
# validation_split=0.1 means that 10% of the training data will be used for validation during training, which helps monitor the model's performance on unseen data and can help prevent overfitting.
# epochs=12 means that the model will go through the entire training dataset 12 times, which allows it to learn from the data and improve its performance up to a point. 
# Note that it will go up to 12 but p[robably less due to callback function to detect overfitting.]
# batch_size=32 means that the model will update its weights after processing every 32 samples, which can help improve training efficiency and convergence. 32 is a common choice.
history = model.fit(
    train_data, train_labels,
    validation_split=0.1,
    epochs=12,
    batch_size=32,
    callbacks=[callback]
)

# Explanation
# For each epoch:
# 1. Shuffle the training data
# 2. Break it into batches of 32
# 3. For each batch:
#       Run a forward pass
#       Compute loss
#       Compute gradients
#       Update weights
# 4. After the epoch, evaluate on validation data
# 5. Store metrics in history

# This repeats for however many epochs I specify.

## 8. Display training and testing results
Plot the training and validation accuracy and print the accuracy of the model on the test data (10k MNIST images).

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.xlabel('Epoch')               # X‑axis label
plt.ylabel('Accuracy')            # Y‑axis label

plt.title('Model Accuracy Over Epochs')
plt.legend()
plt.grid()
plt.show()

# Another helpful explanation from Copilot.
# The history object stores everything the model learned during training:
#   loss per epoch
#   accuracy per epoch
#   validation loss
#   validation accuracy

# Evaluate the model on the test data to see how well it generalizes to unseen data.
test_loss, test_acc = model.evaluate(test_data, test_labels)
print("Test accuracy:", test_acc)


## 9. Save the trained model
Save the model so we can load it later to be used by the frontend.

In [ ]:
model.save("mnist_cnn_model.keras")